# Module 2: Data Quality Checks

## Overview

This notebook performs automated **Data Quality Checks** on the raw retail transaction dataset.

It analyzes the dataset to identify missing values, duplicate records, schema inconsistencies, and other data quality issues. A Data Quality Report is generated to evaluate the condition of the incoming data before the cleaning process begins.

In [0]:
# Load data from Raw Layer

from pyspark.sql.functions import *

raw_df = spark.table("trustguard.raw_transactions")

display(raw_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,true
TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,true
TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,false
TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,null
TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,false
TXN_7482416,CUST_09,Patisserie,null,null,10.0,200.0,Credit Card,Online,2023-11-30,null
TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,true
TXN_1372952,CUST_21,Furniture,null,33.5,null,null,Digital Wallet,In-store,2024-04-02,true
TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,false
TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,false


In [0]:
# Check total records and columns

print("Total Rows:", raw_df.count())
print("Total Columns:", len(raw_df.columns))

Total Rows: 12575
Total Columns: 11


In [0]:
# Null Count Per Column

from pyspark.sql.functions import col, count, when

null_report = raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
])

display(null_report)

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,0,0,1213,609,604,604,0,0,0,4199


In [0]:

# Check duplicate records based on Primary Key (transaction_id)

duplicate_count = (
    raw_df.groupBy("transaction_id")
          .count()
          .filter(col("count") > 1)
          .count()
)

print("Duplicate Transaction IDs:", duplicate_count)

Duplicate Transaction IDs: 0


In [0]:
# Format Mismatch Count

format_mismatch = raw_df.filter(
    col("price_per_unit").cast("double").isNull() |
    col("quantity").cast("int").isNull()
).count()

print("Format Mismatch Count:", format_mismatch)

Format Mismatch Count: 1213


In [0]:
# Referential Integrity Failure Count

referential_failure = raw_df.filter(
    col("transaction_id").isNull() |
    col("customer_id").isNull()
).count()

print("Referential Integrity Failure Count:", referential_failure)

Referential Integrity Failure Count: 0


In [0]:
# Data Quality Summary Report

print("========== DATA QUALITY REPORT ==========")
print("Total Rows:", raw_df.count())
print("Total Columns:", len(raw_df.columns))
print("Duplicate Records:", duplicate_count)
print("Format Mismatch Count:", format_mismatch)
print("Referential Integrity Failure Count:", referential_failure)

display(null_report)

========== DATA QUALITY REPORT ==========
Total Rows: 12575
Total Columns: 11
Duplicate Records: 0
Format Mismatch Count: 1213
Referential Integrity Failure Count: 0


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,0,0,1213,609,604,604,0,0,0,4199
